# Engineer causal same-airline departure-backlog features

Extend the established departure feature dataset with a separate same-airline operational-pressure path. For each sample flight, the scheduled departure timestamp `DATE` is the prediction cutoff. Its trailing cohort contains only flights operated by the same normalized `Reporting_Airline` and scheduled earlier within the configured window. Delay aggregates use only cohort flights that pushed back strictly before the cutoff; earlier-scheduled same-airline flights still waiting to push back contribute only to the pending state.

The existing `data/features/AIRPORT_YEAR_departures.csv` file is read without modification. The output retains every source row and column and appends nine `AIRLINE_BACKLOG_W...` fields. Learned imputation, scaling, encoding, and feature selection remain responsibilities of later model experiments.


In [1]:
YEAR = 2019

AIRPORT = "JFK"

BACKLOG_WINDOW_MINUTES = 60


## Configure the source and output files

The notebook works from either the project root or the `notebooks` directory. The filename records both the same-airline scope and the backlog-window length so it cannot be confused with the established airport-wide W60 dataset.


In [2]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import MODEL_TARGETS
from feature_engineering_backlog import actual_departure_timestamps
from feature_engineering_backlog_airline import (
    add_airline_departure_backlog_features,
    airline_backlog_feature_names,
    validate_airline_departure_backlog_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
BACKLOG_WINDOW_MINUTES = int(BACKLOG_WINDOW_MINUTES)
INPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures.csv"
OUTPUT_FILE = PROJECT_ROOT / (
    f"data/features/{AIRPORT}_{YEAR}_departures_backlog_airline_w"
    f"{BACKLOG_WINDOW_MINUTES}.csv"
)
AIRLINE_BACKLOG_FEATURES = airline_backlog_feature_names(BACKLOG_WINDOW_MINUTES)

print(pd.Series({"input": str(INPUT_FILE), "output": str(OUTPUT_FILE)}))


input     /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
output    /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
dtype: str


## Load and validate the departure population

Validate rather than silently filter the established departure feature dataset. The causal reconstruction requires complete schedule timestamps, completed-flight departure outcomes, and reporting-airline codes. The airline is known from the schedule before the prediction cutoff.


In [3]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Departure feature file does not exist: {INPUT_FILE}")

source = pd.read_csv(INPUT_FILE, low_memory=False)
required_columns = {
    "Year", "Origin", "DATE", "Reporting_Airline", "DepDelay",
    "DepDelayMinutes", "DepDel15", MODEL_TARGETS["1A"],
}
missing_columns = required_columns - set(source.columns)
if missing_columns:
    raise KeyError(f"Departure feature data is missing: {sorted(missing_columns)}")

origin = source["Origin"].astype("string").str.strip().str.upper()
airline = source["Reporting_Airline"].astype("string").str.strip().str.upper()
source_year = pd.to_numeric(source["Year"], errors="coerce")
if not origin.eq(AIRPORT).all():
    raise ValueError(f"Departure input contains origins other than {AIRPORT}")
if not source_year.eq(YEAR).all():
    raise ValueError(f"Departure input contains years other than {YEAR}")
if airline.isna().any() or airline.eq("").any():
    raise ValueError("Reporting_Airline must contain complete airline codes")
if set(AIRLINE_BACKLOG_FEATURES) & set(source.columns):
    raise ValueError("Input already contains same-airline backlog features; use the base departure file")

target = pd.to_numeric(source[MODEL_TARGETS["1A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("DepDel15 must be complete and binary")

airline_counts = airline.value_counts().rename("departures").to_frame()
print(pd.Series({
    "rows": len(source), "columns": len(source.columns),
    "airlines": airline.nunique(), "delayed": int(target.sum()),
}))
airline_counts


rows        107430
columns        112
airlines        10
delayed      20168
dtype: int64


,departures
Reporting_Airline,
B6,33735
DL,31336
AA,15201
9E,14228
AS,4983
MQ,3555
OO,2402
YX,1392
HA,364


## Add the same-airline trailing backlog features

`PENDING_COUNT` is the number of earlier-scheduled same-airline flights in the W60 cohort that had not pushed back by the sample cutoff. Delay counts and summaries use only same-airline flights whose gate-out event was already observable. `PENDING_SHARE` is pending divided by scheduled cohort size. Rates, means, and pending share remain missing when their denominators are zero.


In [4]:
features = add_airline_departure_backlog_features(
    source,
    window_minutes=BACKLOG_WINDOW_MINUTES,
    airline_column="Reporting_Airline",
)
if len(features) != len(source):
    raise ValueError("Same-airline backlog engineering changed the departure row count")
if not features[source.columns].equals(source):
    raise ValueError("Same-airline backlog engineering changed one or more source columns")

feature_validation = validate_airline_departure_backlog_features(
    features,
    window_minutes=BACKLOG_WINDOW_MINUTES,
    airline_column="Reporting_Airline",
)

scheduled_name, completed_name, pending_name = AIRLINE_BACKLOG_FEATURES[:3]
pending_share_name = AIRLINE_BACKLOG_FEATURES[-1]
print(pd.Series({
    "same-airline backlog features": len(AIRLINE_BACKLOG_FEATURES),
    "rows with same-airline pending backlog": int(features[pending_name].gt(0).sum()),
    "rows without completed same-airline history": int(features[completed_name].eq(0).sum()),
    "rows without scheduled same-airline history": int(features[scheduled_name].eq(0).sum()),
    "mean same-airline pending share": features[pending_share_name].mean(),
}))


same-airline backlog features                      9.000000
rows with same-airline pending backlog         36464.000000
rows without completed same-airline history    15998.000000
rows without scheduled same-airline history    13289.000000
mean same-airline pending share                    0.141963
dtype: float64


In [5]:
feature_validation

,dtype,missing_count,missing_percent,minimum,maximum
AIRLINE_BACKLOG_W60_SCHEDULED_COUNT,Int32,0,0.000000,0.0,18.0
AIRLINE_BACKLOG_W60_COMPLETED_COUNT,Int32,0,0.000000,0.0,17.0
AIRLINE_BACKLOG_W60_PENDING_COUNT,Int32,0,0.000000,0.0,12.0
AIRLINE_BACKLOG_W60_DELAYED_DEPARTURE_COUNT,Int32,0,0.000000,0.0,7.0
AIRLINE_BACKLOG_W60_DELAY_RATE,float64,15998,14.891557,0.0,1.0
AIRLINE_BACKLOG_W60_MEAN_DEP_DELAY,float64,15998,14.891557,-21.0,59.0
AIRLINE_BACKLOG_W60_MEAN_DEP_DELAY_MINUTES,float64,15998,14.891557,0.0,59.0
AIRLINE_BACKLOG_W60_TOTAL_DEP_DELAY_MINUTES,float64,0,0.000000,0.0,218.0
AIRLINE_BACKLOG_W60_PENDING_SHARE,float64,13289,12.369915,0.0,1.0


## Validate the prediction cutoff within each airline

Flights operated by the same airline at the same scheduled timestamp must receive identical same-airline backlog state. Flights from different airlines at that timestamp may—and generally should—receive different values. This grouped check confirms that neither a sample outcome nor outcomes from simultaneous same-airline flights enter the trailing cohort.


In [6]:
scheduled_timestamp = pd.to_datetime(features["DATE"], errors="coerce")
actual_timestamp = actual_departure_timestamps(features)
normalized_airline = features["Reporting_Airline"].astype("string").str.strip().str.upper()
if scheduled_timestamp.isna().any() or actual_timestamp.isna().any():
    raise ValueError("Same-airline backlog timing audit found an invalid timestamp")

same_airline_cutoff_variants = (
    features.assign(_CUTOFF=scheduled_timestamp, _AIRLINE=normalized_airline)
    .groupby(["_AIRLINE", "_CUTOFF"], sort=False)[AIRLINE_BACKLOG_FEATURES]
    .nunique(dropna=False)
)
if same_airline_cutoff_variants.to_numpy().max(initial=0) > 1:
    raise ValueError("Same-airline flights at one cutoff received different backlog features")

cutoff_airline_counts = (
    features.assign(_CUTOFF=scheduled_timestamp, _AIRLINE=normalized_airline)
    .groupby("_CUTOFF", sort=False)["_AIRLINE"].nunique()
)
timing_audit = pd.Series({
    "scheduled cutoffs": int(scheduled_timestamp.nunique()),
    "airline-cutoff groups": int(len(same_airline_cutoff_variants)),
    "rows sharing an airline and cutoff": int(
        features.assign(_CUTOFF=scheduled_timestamp, _AIRLINE=normalized_airline)
        .duplicated(["_AIRLINE", "_CUTOFF"], keep=False).sum()
    ),
    "cutoffs containing multiple airlines": int(cutoff_airline_counts.gt(1).sum()),
    "actual departures before their own scheduled time": int(
        (actual_timestamp < scheduled_timestamp).sum()
    ),
    "same-airline cutoff inconsistencies": 0,
}, name="same-airline backlog timing validation")
timing_audit


scheduled cutoffs                                    67723
airline-cutoff groups                                90181
rows sharing an airline and cutoff                   29272
cutoffs containing multiple airlines                 16541
actual departures before their own scheduled time    66834
same-airline cutoff inconsistencies                      0
Name: same-airline backlog timing validation, dtype: int64

## Save the same-airline backlog dataset

Retain every source and audit column, append the nine same-airline fields, and write a separately named dataset. The established departure, airport-wide backlog, and rotation feature files remain intact.


In [7]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT, "year": YEAR,
    "window minutes": BACKLOG_WINDOW_MINUTES,
    "rows": len(features), "columns": len(features.columns),
    "airlines": normalized_airline.nunique(),
    "added columns": len(AIRLINE_BACKLOG_FEATURES),
    "target": MODEL_TARGETS["1A"], "output": str(OUTPUT_FILE),
}, name="same-airline departure backlog feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary


Saved 107,430 rows to /Users/johnkyte/Projects/berkeley_ml_and_ai/capstone/data/features/JFK_2019_departures_backlog_airline_w60.csv


airport                                                         JFK
year                                                           2019
window minutes                                                   60
rows                                                         107430
columns                                                         121
airlines                                                         10
added columns                                                     9
target                                                     DepDel15
output            /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
Name: same-airline departure backlog feature summary, dtype: object